# Revisiting Basics of Nonlinear Spacecraft Control

Spacecraft attitude dynamics are nonlinear, so their stability cannot always be understood reliably using linear approximations alone.

A useful alternative is **Lyapunov's Direct Method**. Instead of solving the equations of motion and examining every possible trajectory, we construct a scalar function

$$
V(\mathbf{x})
$$

that measures how far the system is from a desired equilibrium. We then study how this quantity changes with time through

$$
\dot{V}.
$$

If $V$ behaves like a measure of "stored error" and decreases appropriately as the spacecraft moves, we can make statements about whether the system remains near the equilibrium and whether it eventually converges to it.

This module is primarily a rapid review of the nonlinear stability concepts developed in the first *Spacecraft Dynamics and Control* specialization. It revisits:

- the basic definitions of nonlinear stability,
- the use of Lyapunov functions to prove stability, and
- the interpretation of the signs of $V$ and $\dot V$.

The main extension is the treatment of **non-autonomous systems**, where the system dynamics depend explicitly on time. In this case, the Lyapunov analysis must account for this additional time dependence when constructing and differentiating the candidate function.

The module therefore follows the progression

$$
\boxed{
\text{stability definitions}
\;\rightarrow\;
\text{Lyapunov's Direct Method}
\;\rightarrow\;
\text{time-dependent systems}
}
$$

This notebook focuses only on the concepts and subtleties required for the present course. Full derivations and the more detailed development of nonlinear spacecraft control are available in the notes from the first specialization.

> **Prior material:**  
> [Specialization 1, Course 3 - Nonlinear Controls](https://github.com/johnm3398/Spacecraft-Dynamics-and-Control/tree/main/01_spacecraft_dynamics_and_control_specialization/03_control)
>

---

In [ ]:
import numpy as np

from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt

import sys
sys.path.insert(0, r"../../")
import AttitudeKinematicsLib as ak

In [ ]:
print("Contents of AttitudeKinematicsLib:")
for name in sorted(dir(ak)):
    if not name.startswith("_"):
        print(name)

# 1 - Stability Review

## 1.1 - Stability Definitions & Linearization

In [ ]:
# Quiz 2 - Question 2 

A_11 = -5/8
A_12 = (5 * (6 - 5 * np.sqrt(3))) / (8 * np.sqrt(2))
A_13 = (5 * (6 + 5 * np.sqrt(3))) / (8 * np.sqrt(2))
A_21 = A_12
A_22 = (1 / 16) * (5 - 20 * np.sqrt(3))
A_23 = 115 / 16
A_31 = A_13
A_32 = A_23
A_33 = (1 / 16) * (5 + 20 * np.sqrt(3))

A = np.array([
    [A_11, A_12, A_13],
    [A_21, A_22, A_23],
    [A_31, A_32, A_33]
])

eigenvalues = np.linalg.eigvals(A)

print(eigenvalues.round())

## 1.2 - Lyapunov's Direct Method

## 1.3 - Stability for Time-Dependent Systems

# 2 - Nonlinear Rigid-Body Feedback Control

## 2.1 - Rigid-Body Detumbling

## 2.2 - Lyapunov Functions for the various Attitude Representations

## 2.3 - Three-Axis Attitude Feedback Control

With reference to **Quiz 8**, we numerically simulate the closed-loop attitude motion of a rigid spacecraft governed by

$$
[I]\dot{\boldsymbol{\omega}}_{B/N} =
-[\tilde{\boldsymbol{\omega}}_{B/N}][I]\boldsymbol{\omega}_{B/N}
+
\mathbf{u}.
$$

The spacecraft tracks the prescribed reference attitude
$\boldsymbol{\sigma}_{R/N}(t)$ using

$$
\mathbf{u} =
-K\boldsymbol{\sigma}_{B/R}
-P\boldsymbol{\omega}_{B/R}
+
[I]\left(
\dot{\boldsymbol{\omega}}_{R/N} - [\tilde{\boldsymbol{\omega}}_{B/N}]\boldsymbol{\omega}_{R/N}
\right)
+
[\tilde{\boldsymbol{\omega}}_{B/N}]
[I]\boldsymbol{\omega}_{B/N}.
$$

The propagated state is

$$
\mathbf{x} =
\begin{bmatrix}
\boldsymbol{\sigma}_{B/N}\\
{}^B\boldsymbol{\omega}_{B/N}
\end{bmatrix}.
$$

At each integration step, the reference motion is evaluated, the tracking errors
$\boldsymbol{\sigma}_{B/R}$ and $\boldsymbol{\omega}_{B/R}$ are formed, and the
control torque is applied to the rigid-body equations of motion.

Because $\boldsymbol{\sigma}_{R/N}(t)$ is prescribed directly, the corresponding
reference angular velocity is obtained from the inverse MRP kinematics,

$$
{}^R\boldsymbol{\omega}_{R/N} =
4[B(\boldsymbol{\sigma}_{R/N})]^{-1}
\dot{\boldsymbol{\sigma}}_{R/N}.
$$

The objective is to evaluate the closed-loop attitude tracking error
$\boldsymbol{\sigma}_{B/R}$ at $t=15$ s and $t=40$ s.

In [ ]:
# Time
dt = 0.001
T = 40.0
t = np.arange(0.0, T + dt, dt)
N = len(t)

# Spacecraft inertia matrix
I_1 = 100.0
I_2 = 75.0
I_3 = 80.0
I = np.diag([I_1, I_2, I_3])

# Feedback gains
K = 5.0
P = 10.0

# Reference motion frequency
f = 0.03  # rad/s

# Initial body attitude
sigma_BN_0 = np.array([
    0.1,
    0.2,
    -0.1,
])

# Initial body angular velocity
omega_BN_B_0 = np.deg2rad(np.array([
    30.0,
    10.0,
    -20.0,
]))

# Reference attitude history
sigma_RN_hist = np.column_stack((
    0.1 * np.sin(f * t),
    0.2 * np.cos(f * t),
    -0.3 * np.sin(2.0 * f * t),
))

# Reference MRP rate history
sigma_RN_dot_hist = np.column_stack((
    0.1 * f * np.cos(f * t),
    -0.2 * f * np.sin(f * t),
    -0.6 * f * np.cos(2.0 * f * t),
))

# Reference angular velocity history
omega_RN_R_hist = np.zeros((N, 3))

for k in range(N):
    omega_RN_R_hist[k] = 4.0 * ak.BInvmat_MRP(sigma_RN_hist[k]) @ sigma_RN_dot_hist[k]

# Reference angular acceleration history
omega_dot_RN_R_hist = np.gradient(omega_RN_R_hist, t, axis=0, edge_order=2)

# Initial propagated state
sigma_BN = sigma_BN_0.copy()
omega_BN_B = omega_BN_B_0.copy()

# Logs
sigma_BN_hist = np.zeros((N, 3))
omega_BN_B_hist = np.zeros((N, 3))
sigma_BR_hist = np.zeros((N, 3))
omega_BR_B_hist = np.zeros((N, 3))
u_hist = np.zeros((N, 3))

for k in range(N):

    # Reference motion
    sigma_RN = sigma_RN_hist[k]
    omega_RN_R = omega_RN_R_hist[k]
    omega_dot_RN_R = omega_dot_RN_R_hist[k]

    # Attitude tracking error
    sigma_BR = ak.MRP_compose(sigma_BN, sigma_RN, mode="sub")
    C_BR = ak.MRP_to_DCM(sigma_BR)

    # Reference angular velocity and acceleration expressed in B
    omega_RN_B = C_BR @ omega_RN_R
    omega_dot_RN_B = C_BR @ omega_dot_RN_R

    # Angular velocity tracking error
    omega_BR_B = omega_BN_B - omega_RN_B

    # Skew-symmetric angular velocity matrix
    omega_BN_tilde = ak.skew_symmetric(omega_BN_B)

    # Nonlinear three-axis tracking controller
    u = - K * sigma_BR \
        - P * omega_BR_B \
        + I @ (omega_dot_RN_B - omega_BN_tilde @ omega_RN_B) \
        + omega_BN_tilde @ (I @ omega_BN_B)

    # Rigid-body rotational dynamics
    omega_BN_B_dot = np.linalg.solve(I, -omega_BN_tilde @ (I @ omega_BN_B) + u)

    # MRP kinematics
    sigma_BN_dot = 0.25 * ak.Bmat_MRP(sigma_BN) @ omega_BN_B

    # Log current values
    sigma_BN_hist[k] = sigma_BN
    omega_BN_B_hist[k] = omega_BN_B
    sigma_BR_hist[k] = sigma_BR
    omega_BR_B_hist[k] = omega_BR_B
    u_hist[k] = u

    # Euler integration
    omega_BN_B = omega_BN_B + dt * omega_BN_B_dot
    sigma_BN = sigma_BN + dt * sigma_BN_dot

    # Keep the attitude in the principal MRP set
    sigma_BN = ak.MRP_shadow(sigma_BN)

# Quiz evaluation times
t_1 = 15.0
t_2 = 40.0

index_t1 = np.argmin(np.abs(t - t_1))
index_t2 = np.argmin(np.abs(t - t_2))

sigma_BR_t1 = sigma_BR_hist[index_t1]
sigma_BR_t2 = sigma_BR_hist[index_t2]

print(f"sigma_BR at t = {t_1:.0f} s:", np.round(sigma_BR_t1, 6))
print(f"sigma_BR at t = {t_2:.0f} s:", np.round(sigma_BR_t2, 6))

In [ ]:
# Attitude tracking error
plt.figure(figsize=(8, 4))

plt.plot(t, sigma_BR_hist[:, 0], label=r"$\sigma_{1,B/R}$")
plt.plot(t, sigma_BR_hist[:, 1], label=r"$\sigma_{2,B/R}$")
plt.plot(t, sigma_BR_hist[:, 2], label=r"$\sigma_{3,B/R}$")

plt.axvline(t_1, linestyle="--", alpha=0.5)
plt.axvline(t_2, linestyle="--", alpha=0.5)

plt.title("Attitude Tracking Error")
plt.xlabel("Time [s]")
plt.ylabel("MRP")
plt.grid(True)
plt.legend()
plt.show()


# Angular velocity tracking error
plt.figure(figsize=(8, 4))

plt.plot(t, np.rad2deg(omega_BR_B_hist[:, 0]), label=r"$\omega_{1,B/R}$")
plt.plot(t, np.rad2deg(omega_BR_B_hist[:, 1]), label=r"$\omega_{2,B/R}$")
plt.plot(t, np.rad2deg(omega_BR_B_hist[:, 2]), label=r"$\omega_{3,B/R}$")

plt.title("Angular Velocity Tracking Error")
plt.xlabel("Time [s]")
plt.ylabel("Angular Velocity [deg/s]")
plt.grid(True)
plt.legend()
plt.show()

## 2.4 - Asymptotic Convergence of Attitude Control

# Key Takeaways and Concluding Remarks

---

Notes structured by John G.